<a href="https://colab.research.google.com/github/hakan-karadeniz/Yol-Segmentasyon-Uygulamas-/blob/main/Arayuz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q gradio opencv-python-headless

In [ ]:
import sys
from google.colab import drive
drive.mount('/content/drive')
sys.path.insert(0, '/content/drive/MyDrive/models')

Mounted at /content/drive


In [ ]:
import sys, torch
sys.path.insert(0, '/content')
from shared import UNet, CONFIG

CKPT_PATH = '/content/drive/MyDrive/models/road_best.pth'

device = CONFIG['device']
model  = UNet().to(device)
model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
model.eval();

In [ ]:
import numpy as np
import cv2
from PIL import Image

W, H = CONFIG['image_size']

def predict(pil_img):
    orig_np = np.array(pil_img.convert('RGB'))
    orig_h, orig_w = orig_np.shape[:2]

    resized = cv2.resize(orig_np, (W, H))
    img_f   = resized.astype(np.float32) / 255.0
    tensor  = torch.from_numpy(img_f).permute(2,0,1).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(tensor)
    prob     = torch.sigmoid(logits).squeeze().cpu().numpy()
    prob_full = cv2.resize(prob, (orig_w, orig_h))
    mask_bool = (prob_full > 0.5)

    overlay = orig_np.copy().astype(np.float32)
    overlay[mask_bool, 0] = overlay[mask_bool, 0] * 0.5
    overlay[mask_bool, 1] = overlay[mask_bool, 1] * 0.5 + 255 * 0.5
    overlay[mask_bool, 2] = overlay[mask_bool, 2] * 0.5
    overlay = np.clip(overlay, 0, 255).astype(np.uint8)

    return Image.fromarray(overlay)

In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type='pil', label='Görüntü Yükle'),
    outputs=gr.Image(type='pil', label='Yol Tespiti'),
    title='Yol Segmentasyonu',
    description='Görüntü yükle, yol bölgesi yeşil overlay ile gösterilir.',
)

demo.launch(share=True, show_error=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1380081495b9f2330c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
